In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset
from fundus_vessels_toolkit.segment_to_graph.models.digraph_model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.models.trainer import DigraphGNNTrainer

vscode_theme()


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]
dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, resize_to=1024, root="tmp/DATA", overwrite=False)
train_set, val_set, test_set = dataset.split_loaders(train_ratio=0.7, val_ratio=0.15)

Found 216 branch digraphs...


Processing...
Done!
Preloading dataset: 100%|██████████| 216/216 [00:24<00:00,  8.67it/s]


## Visualize result from pred table


In [4]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


In [5]:
ID = 1

# name, b_parent, b_dir, b_av = parse_arborescence(preds, ID, opti=False)
# m = dataset.show_tree_diff(name, b_parent, b_dir, b_av == -1)
# m

### Load model from checkpoint


In [6]:
model = DigraphGNNTrainer.load_from_checkpoint("last.ckpt").model.cuda()


In [47]:
ID = 5
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(val_set.get(ID).cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir = out.optimal_tree
m, pred_tree = val_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
)
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">GT Tree: g_035</h3>'), HTML(value='<h3 style="te…

In [34]:
out.max_parent()[107]

tensor(274, device='cuda:0')

In [10]:
out.gt_dir_p.mean()

tensor(0.5077, device='cuda:0')

In [11]:
out.max_parent()[183]

tensor(104, device='cuda:0')

In [12]:
out.dir_logit[183]

tensor(0.6776, device='cuda:0')

In [13]:
out.final_lines_p[out.lines.b1 == 197]

tensor([4.6443e-27, 1.1056e-27, 3.6979e-25, 2.1224e-19, 5.9323e-15, 2.3590e-31,
        9.0399e-25, 2.6988e-19, 1.6510e-19, 2.3616e-15, 2.7396e-09, 9.6394e-01,
        2.9690e-05, 4.6609e-18, 1.4442e-07, 8.4449e-17, 2.7366e-13, 3.9536e-04,
        8.9612e-23, 8.2357e-38, 1.8344e-11, 3.5125e-02, 5.1274e-04, 2.4649e-09],
       device='cuda:0')

In [14]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_by_branch(b1=197, sort_by_p=True)

array([[131.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ -1.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [182.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 89.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [175.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [163.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [102.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 88.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 68.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 67.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 70.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 84.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 76.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 91.  ,   0.  , 197.  ,   1.  ,   0.  ,   1.  ,   1.  ],
       [ 80.  ,   0.  , 197.  ,   1.  ,   0.  ,   0.99,   1.  ],
       [114.  ,   1.  , 1

In [15]:
test_set.get_sample(ID)[0].graph.branch_list[[183]]

array([[187, 108]])

In [16]:
out.lines.edge_first_tip[(out.lines.b0 == 15) & (out.lines.b1 == 14)]

tensor([[True, True]], device='cuda:0')

In [28]:
from torch import gt
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

pred_parent_acc = []
pred_dir_acc = []
opti_parent_acc = []
opti_dir_acc = []
baseline_parent_acc = []
baseline_dir_acc = []

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(val_set))):
        digraph, _, od_yx, _ = val_set.get_sample(i)
        valid_branch = ~digraph.missing_branch()
        od = Point.parse(od_yx)
        assert digraph.graph is not None, "Graph must be loaded to infer tree"

        art_branch = digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        out = model(val_set.get(i).cuda())
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        pred_parent_acc.append((pred_parent == digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_parent, opti_dir = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (digraph.branch_dir_p[valid_branch] > 0.5)).mean())

100%|██████████| 32/32 [00:14<00:00,  2.17it/s]


In [29]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.917576122156633),
 np.float64(0.9237345350338383),
 np.float64(0.8479755095544518))

In [30]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.9863569949372907),
 np.float64(0.9887057766137222),
 np.float64(0.9428590582355942))